In [ ]:
import h5py
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

print("TensorFlow version:", tf.__version__)
print("All libraries loaded!")

In [ ]:
# Paths - adjust them! 
NOISE_HDF5 = r'/your/path'
EQ_HDF5    = r'/your/path'
NOISE_CSV  = r'/your/path'
EQ_CSV     = r'/your/path'

WAVE_LENGTH = 6000   # STEAD waveforms = 6000 samples (60 sec at 100Hz)
N_EACH      = 1000   # 1000 noise + 1000 earthquake = 2000 total

def load_stead_samples(hdf5_path, csv_path, n_samples, label):
    X, y = [], []

    # Read CSV to get trace names
    df = pd.read_csv(csv_path)
    print(f"CSV loaded — {len(df)} total entries")

    # Sample n_samples rows
    df = df.sample(n=min(n_samples, len(df)), random_state=42)

    with h5py.File(hdf5_path, 'r') as f:
        for _, row in df.iterrows():
            try:
                trace_name = row['trace_name']
                waveform   = f['data'][trace_name][:]   # shape: (6000, 3)

                # Use vertical (Z) channel only — index 2
                channel = waveform[:, 2].astype(np.float32)

                # Normalize by standard deviation
                channel = channel / (np.std(channel) + 1e-8)

                X.append(channel)
                y.append(label)
            except Exception as e:
                continue   # skip corrupted entries silently

    print(f"Loaded {len(X)} samples with label={label}")
    return X, y

# Load both classes
print("Loading noise waveforms...")
X_noise, y_noise = load_stead_samples(NOISE_HDF5, NOISE_CSV, N_EACH, label=0)

print("\nLoading earthquake waveforms...")
X_eq, y_eq = load_stead_samples(EQ_HDF5, EQ_CSV, N_EACH, label=1)

# Combine
X = np.array(X_noise + X_eq)[..., np.newaxis]   # shape: (2000, 6000, 1)
y = np.array(y_noise + y_eq)

# Shuffle
idx  = np.random.permutation(len(X))
X, y = X[idx], y[idx]

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nFinal dataset:")
print(f"Input shape    : {X.shape}")
print(f"Earthquakes    : {np.sum(y==1)}")
print(f"Noise          : {np.sum(y==0)}")
print(f"Train          : {len(X_train)}")
print(f"Test           : {len(X_test)}")

In [ ]:
eq_idx    = np.where(y == 1)[0][0]
noise_idx = np.where(y == 0)[0][0]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 5))

ax1.plot(X[eq_idx, :, 0], color='#E24B4A', linewidth=0.6)
ax1.set_title('Real earthquake waveform — label = 1 (STEAD chunk2)')
ax1.set_xlabel('Time steps (100Hz → 6000 = 60 seconds)')
ax1.set_ylabel('Normalized amplitude')
ax1.grid(True, alpha=0.3)

ax2.plot(X[noise_idx, :, 0], color='#378ADD', linewidth=0.6)
ax2.set_title('Real noise waveform — label = 0 (STEAD chunk1)')
ax2.set_xlabel('Time steps (100Hz → 6000 = 60 seconds)')
ax2.set_ylabel('Normalized amplitude')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
print(f"Training samples : {len(X_train)}")
print(f"Test samples     : {len(X_test)}")
print(f"Input shape      : {X_train.shape}")

In [ ]:
model = models.Sequential([

    # CONVOLUTIONAL LAYER 1
    layers.Conv1D(32, kernel_size=5, activation='relu',
                  input_shape=(6000, 1)),
    layers.BatchNormalization(),
    layers.MaxPooling1D(pool_size=4),

    # CONVOLUTIONAL LAYER 2
    layers.Conv1D(64, kernel_size=3, activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling1D(pool_size=4),

    # CONVOLUTIONAL LAYER 3
    layers.Conv1D(128, kernel_size=3, activation='relu'),
    layers.MaxPooling1D(pool_size=4),

    # FLATTEN
    layers.Flatten(),

    # FULLY CONNECTED
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.5),

    # OUTPUT
    layers.Dense(1, activation='sigmoid')
])

model.summary()

In [ ]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)
print("Model compiled!")

In [ ]:
lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

history = model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=64,        # 64 for faster training on CPU
    validation_split=0.1,
    callbacks=[lr_scheduler],
    verbose=1
)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(history.history['loss'],     label='Train loss',     color='#E24B4A')
ax1.plot(history.history['val_loss'], label='Val loss',       color='#E24B4A',
         linestyle='--', alpha=0.6)
ax1.set_title('Loss over epochs')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(history.history['accuracy'],     label='Train accuracy', color='#1D9E75')
ax2.plot(history.history['val_accuracy'], label='Val accuracy',   color='#1D9E75',
         linestyle='--', alpha=0.6)
ax2.set_title('Accuracy over epochs')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
y_pred_prob = model.predict(X_test)
y_pred      = (y_pred_prob > 0.5).astype(int)

print("=== Classification Report ===")
print(classification_report(y_test, y_pred,
      target_names=['Noise', 'Earthquake']))

print("=== Confusion Matrix ===")
print(confusion_matrix(y_test, y_pred))

In [ ]:
print(f"Test set size: {len(X_test)} samples")
print(f"Valid indices: 0 to {len(X_test)-1}\n")

idx        = 0
sample     = X_test[idx : idx+1]
true_label = y_test[idx]

prob      = model.predict(sample, verbose=0)[0][0]
predicted = "Earthquake" if prob > 0.5 else "Noise"
actual    = "Earthquake" if true_label == 1 else "Noise"

print(f"Probability score : {prob:.4f}")
print(f"Predicted         : {predicted}")
print(f"Actual            : {actual}")
print(f"Correct?          : {'Yes' if predicted == actual else 'No'}")